# 03 - 订阅 crewai_event_bus

CrewAI 0.20+ 把所有执行事件发布到 `crewai_event_bus` 上。你可以在
不修改 crew 内部代码的情况下,订阅这些事件来观察 / 记录 / 触发副作用。

常见事件类型:

- `CrewKickoffStarted` / `CrewKickoffCompleted`
- `TaskStarted` / `TaskCompleted` / `TaskFailed`
- `AgentExecutionStarted` / `AgentExecutionCompleted`
- `ToolUsageStarted` / `ToolUsageFinished`
- `LLMCallStarted` / `LLMCallCompleted`


In [ ]:
from crewai.events import (
    crewai_event_bus,
    TaskStartedEvent,
    TaskCompletedEvent,
    ToolUsageStartedEvent,
    ToolUsageFinishedEvent,
)

def on_task_start(source, event):
    desc = getattr(event, "context", "") or ""
    print(f"  ▶ TASK START: {desc[:80]}")

def on_task_done(source, event):
    print("  ✓ TASK DONE")

def on_tool_start(source, event):
    print(f"    ↪ TOOL: {event.agent_role} → {event.tool_name} args={event.tool_args}")

def on_tool_done(source, event):
    print(f"    ✓ TOOL DONE: {event.tool_name}")

# crewai_event_bus.on(EventType)(handler) 返回装饰器,
# 但直接调用也可以注册 handler。
crewai_event_bus.on(TaskStartedEvent)(on_task_start)
crewai_event_bus.on(TaskCompletedEvent)(on_task_done)
crewai_event_bus.on(ToolUsageStartedEvent)(on_tool_start)
crewai_event_bus.on(ToolUsageFinishedEvent)(on_tool_done)
print("事件订阅已安装")


In [ ]:
from alphaquant.infrastructure.llm import get_llm
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool

@tool("reverse")
def reverse_tool(text: str) -> str:
    """反转字符串。"""
    return text[::-1]

llm = get_llm(temperature=0.1)
agent = Agent(
    role="反转员",
    goal="反转用户输入",
    backstory="用 reverse 工具反转字符串。",
    tools=[reverse_tool],
    llm=llm,
    verbose=False,
)
task = Task(
    description="反转字符串 'CrewAI'",
    expected_output="反转后的字符串",
    agent=agent,
)
crew = Crew(
    agents=[agent],
    tasks=[task],
    process=Process.sequential,
    verbose=False,
)
print("Crew ready")


In [ ]:
print("=== Kickoff ===")
result = crew.kickoff(inputs={})
print("=== Done ===")
print(f"Final: {result.raw}")


## 这就是 `infrastructure/crew_events.py` 在做的事

那个文件实现了:

- `CrewEventBridge`:一个线程安全的事件订阅器,把事件存到 deque
- `CrewEventSnapshot`:UI 读取的不可变快照
- `format_event_line`:把事件格式化成人类可读的单行

Streamlit `1_Analyze.py` 页面用它来:

1. 在 status 块内显示"当前正在运行 agent X 调用 tool Y"
2. 在 `asyncio.run` 完成后,在折叠块里 dump 整个事件时间线

你可以把这里的 `print` 换成 `logging.info` 或 `bridge._append()`,
就构成了生产代码的 `CrewEventBridge` 雏形。
